# Experiment 006: reportable 0.5B, frozen protocol, seeds 1 and 2 (Colab)

Seed 0 ran 15 jul, seed 1 ran 18 jul. This session does two things, in order:
first the cheap generative evals for the 007/008 seed 1 and 2 checkpoints
(makes the supervised headline three-seeded and tests whether supervised
endpoints are stable where grpo's 14.7pt seed gap says rl is not), then 006
seed 2, the adaptive replication triggered by that gap. Everything is
resume-safe, finished work skips.

## Seed reuse map

Seed 2 reuses the shared 006 base eval (base_eval.json, base_logprob.json in
base_evals/006_COLAB_FULL_B05) exactly like seed 1 did, and trains its own sft
and grpo stages from scratch. The 007/008 gen evals only load saved
checkpoints, they train nothing.

## 0. Compute units and mount Drive

Fill `cu_before` now. The open record is saved straight after Drive mounts,
so a disconnected session can be closed when the next runtime starts.

In [ ]:
# note the balance before anything burns units, then save it after drive mounts
import glob, json, os, time
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 439.93   # e.g. 87.3
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

# close an unfinished session with the balance captured at this runtime start
old_sessions = [p for p in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not p.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None:
    print('reminder: fill cu_before, then rerun this cell to save the session')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])

    if not os.path.exists(open_path):
        start = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                 'notebook': '006_seed2_and_supervised_gen_evals', 'colab_cu_before': cu_before}
        with open(open_path, 'w') as f:
            json.dump(start, f, indent=2)
        print('saved open session:', session_id)
print('drive mounted ->', ROOT)

session 20260719_102303 | cu_before = FILL ME
Mounted at /content/drive
reminder: fill cu_before, then rerun this cell to save the session
drive mounted -> /content/drive/MyDrive/inm363


## 1. Check the GPU


In [2]:
!nvidia-smi -L
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU runtime is required')
gpu_name = torch.cuda.get_device_name(0)
print('cuda:', torch.cuda.is_available(), '| torch:', torch.__version__, '| gpu:', gpu_name)
if 'A100' not in gpu_name:
    raise RuntimeError(f'A100 required for experiment 006 seed 1, found: {gpu_name}')


GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-58ab8030-871a-571a-786a-0becd8d3e010)
cuda: True | torch: 2.11.0+cu128 | gpu: NVIDIA A100-SXM4-40GB


## 2. Clone the repo (automatic)

This cell clones the main branch of the public repository without asking for a GitHub token. For online experiment tracking, add `WANDB_API_KEY` under the key icon in Colab Secrets and grant this notebook access. If using a private fork, set `REPO_SLUG` to its owner and name, then add a read-only `GH_TOKEN` for that fork. A missing token for a private fork triggers a one-time prompt.


In [3]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    # pull the latest code when a resumed runtime already has the repo
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())


Cloning into 'INM363-IndividualProject-Dissertation'...
remote: Enumerating objects: 1347, done.
remote: Counting objects: 100% (604/604), done.
remote: Compressing objects: 100% (390/390), done.
remote: Total 1347 (delta 332), reused 477 (delta 211), pack-reused 743 (from 1)
Receiving objects: 100% (1347/1347), 37.41 MiB | 18.78 MiB/s, done.
Resolving deltas: 100% (751/751), done.
working in: /content/INM363-IndividualProject-Dissertation/001_Code


## 3. Install dependencies

Keep colabs own cuda torch, just pin the libraries that drifted.


In [4]:
!pip install -q trl==1.7.1 transformers==5.5.4 peft==0.19.1 accelerate==1.14.0 datasets==4.8.4 wandb pyyaml
# remove torchao because this run does not use quantisation and it conflicts with PEFT
!pip uninstall -q -y torchao
print('deps installed')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 166.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 55.7 MB/s eta 0:00:00
deps installed


## 4. W&B key (automatic)

Reads `WANDB_API_KEY` from Colab Secrets and writes it to the file the code expects.
If the secret isn't set, it falls back to a manual upload of `wandb_api_key.txt`.


In [5]:
import os

# use the local key file first, then Colab Secrets, then a manual upload
if os.path.exists('wandb_api_key.txt'):
    print('wandb_api_key.txt already present, using it')
else:
    wkey = None
    try:
        from google.colab import userdata
        wkey = userdata.get('WANDB_API_KEY')
    except Exception:
        wkey = None
    if wkey:
        open('wandb_api_key.txt', 'w').write(wkey.strip())
        print('wrote wandb key from colab secret')
    else:
        from google.colab import files
        files.upload()  # I pick wandb_api_key.txt here
print('have key file:', os.path.exists('wandb_api_key.txt'))


wrote wandb key from colab secret
have key file: True


## 5. Build the data splits

Materialises gsm8k (sft), mmlu dev+val (grpo) and mmlu test (eval), and checks for leakage.


In [6]:
!python scripts/build_data.py


README.md: 100% 7.93k/7.93k [00:00<00:00, 22.0MB/s]

main/train-00000-of-00001.parquet: downloading bytes:   4% 84.5k/2.31M [00:01<00:34, 63.8kB/s]
main/train-00000-of-00001.parquet: downloading bytes: 100% 2.30M/2.30M [00:01<00:00, 1.68MB/s,  224kB/s  ]
main/train-00000-of-00001.parquet: reconstructing file: 100% 2.31M/2.31M [00:01<00:00, 1.68MB/s,  224kB/s  ]

main/test-00000-of-00001.parquet: downloading bytes:   0% 0.00/419k [00:00<?, ?B/s]
main/test-00000-of-00001.parquet: downloading bytes: 100% 419k/419k [00:01<00:00, 320kB/s, 40.8kB/s  ]
main/test-00000-of-00001.parquet: reconstructing file: 100% 419k/419k [00:01<00:00, 320kB/s, 40.8kB/s  ]
Generating train split: 100% 7473/7473 [00:00<00:00, 397565.12 examples/s]
Generating test split: 100% 1319/1319 [00:00<00:00, 272741.42 examples/s]
README.md: 100% 53.2k/53.2k [00:00<00:00, 77.5MB/s]
dataset_infos.json: 100% 138k/138k [00:00<00:00, 159MB/s]

high_school_psychology/test-00000-of-000(…): downloading bytes:   0% 0.00/92.8k [00

## 6. Wiring check

Quick sanity that the config loads, the gpu is seen, and the trackers resolve.


In [7]:
!python scripts/run.py --config configs/006_colab_full_b05_seed2.yaml

env=COLAB mode=FULL scale=B05 condition=SFT_GRPO
model=Qwen/Qwen2.5-0.5B device=cuda stages=['base_eval', 'sft', 'sft_eval', 'grpo', 'grpo_eval']
steps=500 N=8 seed=2
trackers=['tensorboard', 'wandb']
results -> /content/drive/MyDrive/inm363/results/006_COLAB_FULL_B05_sft_grpo_seed2/  checkpoints -> /content/drive/MyDrive/inm363/checkpoints/006_COLAB_FULL_B05_sft_grpo_seed2/
local smoke wiring ok


## 7a. Supervised gen evals, seeds 1 and 2 (007 p8 + 008, both scales)

8 evals of saved checkpoints, ~1.5h at the observed 16-token eval speeds.
Finished evals skip in seconds if the session reconnects.

In [8]:
# python loop with ! calls so the logs stream live
D = '/content/drive/MyDrive/inm363'
jobs = []
for sc, suffix in (('B05', 'b05'), ('B3', 'b3')):
    for s in (1, 2):
        jobs.append((f'007_p8_COLAB_FULL_{sc}_sft_only_seed{s}', f'configs/007_p8_colab_{suffix}.yaml'))
        jobs.append((f'008_wsft_COLAB_FULL_{sc}_sft_only_seed{s}', f'configs/008_wsft_colab_{suffix}.yaml'))
for tag, cfg in jobs:
    print(f'===== gen eval {tag} =====')
    !python scripts/eval_adapter.py --config {cfg} --adapter {D}/checkpoints/{tag}/sft --condition sft --out {D}/results/{tag}

===== gen eval 007_p8_COLAB_FULL_B05_sft_only_seed1 =====
config.json: 100% 681/681 [00:00<00:00, 3.13MB/s]
tokenizer_config.json: 100% 7.23k/7.23k [00:00<00:00, 11.9MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 111MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 116MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 135MB/s]

model.safetensors: downloading bytes:  12% 117M/988M [00:02<00:08, 97.8MB/s, 9.16MB/s  ] 
model.safetensors: reconstructing file:  14% 136M/988M [00:02<00:17, 49.9MB/s]
model.safetensors: reconstructing file:  15% 147M/988M [00:02<00:15, 52.6MB/s, 12.8MB/s  ]
model.safetensors: downloading bytes:  14% 137M/988M [00:02<00:10, 84.2MB/s, 11.7MB/s  ]
model.safetensors: downloading bytes:  40% 397M/988M [00:03<00:01, 459MB/s, 28.4MB/s  ]
model.safetensors: downloading bytes:  57% 560M/988M [00:03<00:00, 625MB/s, 42.3MB/s  ]
model.safetensors: downloading bytes:  70% 688M/988M [00:03<00:00, 632MB/s, 55.3MB/s  ]
model.safetensors: downloading bytes:  77% 759M/988M [

## 7b. Full run, 006 seed 2

The adaptive replication: same frozen protocol, same a100/bf16 runtime, only
the seed changes. ~4h.

In [9]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/run.py --config configs/006_colab_full_b05_seed2.yaml --stage all

Streaming output truncated to the last 5000 lines.
│ │ is referred   │ requires the  │               │              │           │ │
│ │ to as:        │ audience to   │               │              │           │ │
│ │ A.            │ understand    │               │              │           │ │
│ │ Engagement.   │ and evaluate  │               │              │           │ │
│ │ B. Responses. │ the benefits  │               │              │           │ │
│ │ C.            │ or            │               │              │           │ │
│ │ Advertisemen… │ expectations  │               │              │           │ │
│ │ D. Publicity. │ of the        │               │              │           │ │
│ │ <think>       │ campaign.     │               │              │           │ │
│ │               │ Therefore,    │               │              │           │ │
│ │               │ all of the    │               │              │           │ │
│ │               │ above are     │               │       

## 8. Peek at the results

Everything is saved under results/<tag>/ so I can re-analyse without rerunning.


In [10]:
import json, os
from config import load_config, run_tag
from scripts.run import base_eval_dir

# show both seeds so it is clear what finished and what is still missing
for config_name in ('006_colab_full_b05_seed0.yaml', '006_colab_full_b05_seed1.yaml', '006_colab_full_b05_seed2.yaml'):
    cfg = load_config(os.path.join('configs', config_name))
    tag = run_tag(cfg)
    result_dir = os.path.join(cfg.output_root, 'results', tag)
    print('\nrun:', tag)
    print('results dir:', result_dir)
    print('files:', os.listdir(result_dir) if os.path.isdir(result_dir) else 'missing')
    for name in ('summary', 'manifest'):
        path = os.path.join(result_dir, f'{name}.json')
        if os.path.exists(path):
            print(f'\n{name}:')
            print(json.dumps(json.load(open(path)), indent=2)[:2500])
        else:
            print(f'{name}.json not there yet')

seed1_cfg = load_config('configs/006_colab_full_b05_seed1.yaml')
shared_base = base_eval_dir(seed1_cfg)
print('\nshared base evals:', os.listdir(shared_base) if os.path.isdir(shared_base) else 'missing')



run: 006_COLAB_FULL_B05_sft_grpo_seed0
results dir: /content/drive/MyDrive/inm363/results/006_COLAB_FULL_B05_sft_grpo_seed0
files: ['sft_trainlog.json', 'sft_preds.jsonl', 'sft_eval.json', 'sft_logprob_preds.jsonl', 'sft_logprob.json', 'grpo_trainlog.json', 'grpo_eval.json', 'grpo_preds.jsonl', 'grpo_logprob_preds.jsonl', 'grpo_logprob.json', 'summary.json', 'manifest.json', 'run.log']

summary:
{
  "scale": "B05",
  "mode": "FULL",
  "condition": "SFT_GRPO",
  "seed": 0,
  "base_acc": {
    "easy": 0.20819112627986347,
    "medium": 0.18719211822660098,
    "hard": 0.13131313131313133
  },
  "sft_acc": {
    "easy": 0.21843003412969283,
    "medium": 0.2200328407224959,
    "hard": 0.1691919191919192
  },
  "grpo_acc": {
    "easy": 0.5597269624573379,
    "medium": 0.4630541871921182,
    "hard": 0.24494949494949494
  },
  "reference": "sft",
  "grpo_delta": {
    "medium": 0.2430213464696223,
    "hard": 0.07575757575757575,
    "easy": 0.341296928327645
  },
  "base_logprob_acc": 

## 9. Close the units record

Seed 0 has no historical closing balance. Fill `cu_after` for seed 1 when the run
has finished.


In [11]:
# closing balance, completes the open record for this session
import json, os, time
cu_after = 404.61    # fill from the resources panel again
if cu_after is None:
    print('reminder: fill cu_after for the record')
elif not os.path.isdir('/content/drive/MyDrive'):
    print('drive not mounted, run the mount cell first')
else:
    os.makedirs(OPEN_UNITS, exist_ok=True)
    open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
    if not os.path.exists(open_path):
        print('no open record for this session, it may already be closed')
    else:
        with open(open_path) as f:
            rec = json.load(f)
        rec['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        rec['colab_cu_after'] = cu_after
        rec['colab_cu_used'] = round(rec['colab_cu_before'] - cu_after, 2)
        rec['closed_on_resume'] = False
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(rec) + '\n')
        os.remove(open_path)
        print('logged:', rec)

reminder: fill cu_after for the record
